[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/bending/bending.ipynb)

# Euler–Bernoulli Beam Bending

This example computes how a beam bends under a load, in Euler–Bernoulli beam theory: as the beam bends, each of its cross-sections stays flat and turns a little about a line in its own plane, and the material on one side of that line is stretched while the material on the other side is squeezed.

The computation is written in the projective geometric algebra of three-dimensional space, in which points, lines and planes, forces along lines, and the motions of rigid bodies are all elements of one algebra, and a beam's stiffness is a map between them.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import PGA3D
from examples.animation import save_figure, show_animation
from examples.mechanics.bending import render
from examples.mechanics.bending.core import point
from examples.mechanics.bending.mesh import NAMES, section_meshes
from examples.mechanics.simplex import barycentric_samples, simplex_inertia_weights

In [ ]:
ga = PGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Plane = ga.gatype.vector()
Point = ga.gatype.antivector()
Line = ga.gatype.bivector()
Motor = ga.gatype.rotor()
# Lines in the section's plane.
Strain = ga.gatype(ga.subspace("wz yz zx"))
# Lines along the beam, and couples across it.
Resultant = ga.gatype(ga.subspace("xy wx wy"))
# A strain to the resultant of the fibres' pulls.
Stiffness = ga.gatype((Resultant, Strain))                 # Resultant <- Strain

# The cross-section at the clamp, the point at infinity along the beam, and the point at infinity up the
# section.
section_plane = mv.z                                       # [] Plane
beam_direction = section_plane.dual()                      # [] Point
up_direction = mv.y.dual()                                 # [] Point

# The material's modulus; the beam's length, the pieces it is cut into, and the load at its tip.
modulus = 1.0
length = 10.0
pieces = 40
load = 4e-5
# The frames of the load's turn, and the duration of each.
frames_count = 48
duration_ms = 60
# The size of a push along the beam, and the points it goes through, across and up: one inside every
# section's kern and one outside. The half width of the view of the kern, and its grid.
push = 1e-3
push_points = [[0.03, 0.04], [0.1, 0.3]]
reach = 0.75
kern_grid = 161

In [ ]:
# The example cross-sections, as triangles.
sections = section_meshes()
save_figure(render.draw_meshes(sections), "bending_meshes");

## 1. The stiffness of a cross-section

The beam is modelled as a bundle of fibres: a fibre is a point of the cross-section extruded along the length of the beam, and each fibre stretches or shortens on its own. When the beam bends, each cross-section stays flat and turns a little, relative to its neighbour, about a line in its own plane, the neutral axis; the fibres on one side of that line are stretched and those on the other side squeezed, in proportion to their distance from it.

In projective geometric algebra a turn about a line is described by the line itself, with the line's size the rate of turning, and a line at infinity describes a shift instead of a turn. How one cross-section moves relative to the next, per unit length of beam, is therefore a single line in the section's plane, `Strain`, sized by how fast the sections turn about it: the neutral axis. A shift along the beam, which stretches every fibre alike, moves that line within the plane, and a pure shift puts it at infinity. This line is the strain of the section.

In the algebra a fibre is the line along the beam through its point, `beam_direction & point`, and the regressive product `strain & fibre` is its stretch: zero for a fibre that meets the neutral axis, and growing with its distance from it. A stretched fibre pulls along itself in proportion to its stretch, `modulus * fibre * (strain & fibre)`. With the strain left open this is a map, and summed over the section it is the section's stiffness, from a strain to the resultant of all the fibres' pulls, their combined pull as a single line of force.

The sum over the fibres is an integral over the section's area, computed by an exact quadrature for second moments over a simplex, the same in any number of dimensions: a uniform triangle has the same second moment as three equal point masses inside it, as the [simplex example](../simplex.py) computes and checks against sampling.

In [ ]:
def areas(triangles: Point) -> Scalar:
    """Each triangle's area, from the plane through its three corners."""
    return (triangles[..., 0] & triangles[..., 1] & triangles[..., 2]).norm() / 2


def centroid(triangles: Point) -> Point:
    """The centroid of the whole section, one point for all its triangles: the mean of each triangle's corners,
    weighted by the triangle's area."""
    return (areas(triangles)[..., None] * triangles / 3).sum(axis=(-2, -1)).normalized()


def stiffness(triangles: Point, modulus: float) -> Stiffness:
    """Sum the fibres' pulls under a strain, integrating each triangle exactly."""
    # Per triangle, the points halfway between each corner and the centroid, each with a third of its area.
    weights = simplex_inertia_weights(3)
    samples = barycentric_samples(weights, triangles) * areas(triangles).square_root()[..., None]   # [triangles, samples] Point
    fibres = beam_direction & samples                                      # [triangles, samples] Resultant
    stretches = Strain & fibres                                            # [triangles, samples] Scalar <- Strain
    # Each point carries the square root of its share of the area and appears twice, so its term counts
    # that share once.
    tensions = modulus * fibres * stretches                                # [triangles, samples] Resultant <- Strain
    return tensions.sum(axis=(-2, -1))                                     # [] Resultant <- Strain


triangles, boundary = sections[NAMES.index("Z-section")]                   # [triangles, 3], [boundary] Point
stiffness(triangles, modulus)

The Z-section's stiffness holds its area and its first and second moments of area, all from the one sum, with no centroid found first and no lever arms. Its input, `Strain`, is the lines in the section's plane; its output, `Resultant`, is the lines along the beam, the pulls of the fibres, and the couples across it, pure turning effects, which a pair of opposite pulls on parallel fibres makes.

## 2. Cantilevers under a load at the tip

Each beam is a cantilever: clamped at one end, `z = 0`, and loaded straight down at the other end, its tip, through the centroid of the section there. The beam is cut into pieces along its length, and the strain is computed at the middle of each piece.

A cross-section holds up everything between it and the tip, so its fibres must supply the turning effect of the load about that section, the load's couple, which grows with the section's distance from the tip. In the algebra the load is a line, through the tip along the load's direction. To find its couple about a section, the load line is shifted back along the beam by the section's distance from the clamp, so that the section lies in the plane of the clamp; the point where the shifted line meets that plane, joined with the point at infinity along the beam, is the line of force the fibres must supply. For a load across the beam, the load line is parallel to the plane and meets it at infinity, and that line of force is a couple: the load times its distance from the section. Solving the stiffness against it gives the strain.

The strains then bend the beam. An unbent beam carries each cross-section to the next by a shift along the beam, and a shift is a turn about a line at infinity: here the line where the section's plane meets the plane at infinity, `shift_along_beam = section_plane ^ w`. Bending adds the strain. Within a piece, each cross-section moves relative to the next, per unit length, by `shift_along_beam + strain`: shifted along the beam, and turned about the neutral axis. Over the length of the piece this motion adds up to a rigid motion, its exponential, a motor; the motors of the pieces, multiplied in turn from the clamp outwards, place every cross-section of the bent beam.

The couples are measured on the straight beam, which holds for small deflections. The strain turns no section about the beam's own axis, since the fibres measure stretch only: torsion, which a load across an open section such as the slotted tube also causes, is not part of this stiffness.

In [ ]:
# The line where the section's plane meets the plane at infinity: a turn about it shifts a section along the
# beam, the motion from one cross-section to the next of the unbent beam, per unit length.
shift_along_beam = section_plane ^ mv.w                                      # [] Line


def carried(loads: Line, stations: np.ndarray) -> Resultant:
    """What the fibres carry at each station along the beam: the load seen from the station, met with
    the section's plane, and joined with the point at infinity along the beam."""
    shifts = (shift_along_beam * (stations / 2)).exp()                       # [stations] Motor
    return beam_direction & ((shifts << loads[..., None]) ^ section_plane)   # [..., stations] Resultant


def bend(strains: Strain, step: float) -> Motor:
    """The frame of each cross-section from the clamp to the tip `[..., pieces + 1]`."""
    # Integrate each constant strain exactly; this does not update the load on the bent geometry.
    twists = ((shift_along_beam + strains) * (step / 2)).exp()               # [..., pieces] Motor
    clamp = mv.rotor().broadcast_to(twists.shape[:-1] + (1,))                # [..., 1] Motor
    # Each later twist multiplies from the right: the reverse of the running product of the reverses.
    return concatenate([clamp, twists.reverse().cumprod(axis=-1).reverse()], axis=-1)   # [..., pieces + 1] Motor


def stress(strains: Strain, modulus: float, points: Point) -> Scalar:
    """The stress along the beam at points of the section: the modulus times each fibre's stretch."""
    return modulus * (strains & (beam_direction & points))                   # [...] Scalar


def cantilever(triangles: Point, loads: Point) -> tuple[Strain, Motor, Strain, Scalar]:
    """A cantilever with its tip load through the centroid, batched over loads: the strain of each piece,
    the frame of each cross-section, the principal bending axes, and the stress at the clamp."""
    step = length / pieces
    # The middle of each piece.
    stations = (np.arange(pieces) + 0.5) * step                             # [pieces]
    rigidity = stiffness(triangles, modulus)                                 # [] Resultant <- Strain
    # The centroid of the whole section, and the tip of the unbent beam: the centroid shifted along the
    # beam by the beam's length.
    centre = centroid(triangles)                                             # [] Point
    tip = (shift_along_beam * (length / 2)).exp() >> centre                 # [] Point
    # Lever arms are measured on the straight reference beam.
    strains = rigidity.solve(carried(tip & loads, stations))                # [..., pieces] Strain
    frames = bend(strains, step)                                             # [..., pieces + 1] Motor
    # Principal bending axes pass through the centroid; the metric does not see the shift along the beam.
    _, axes = (Strain | Strain).eigh(Strain & rigidity)                      # [3] Strain
    stresses = stress(strains[..., 0, None, None], modulus, triangles)       # [..., triangles, 3] Scalar
    return strains, frames, axes[:2], stresses


beams = [cantilever(triangles, load * -up_direction) for triangles, _ in sections]
save_figure(render.draw_beams(sections, load * -up_direction, beams), "bending_sections");

The top row shows each section at the clamp, coloured by the stress along the beam: the load in orange, the direction the tip moves in blue, the neutral axis in green, and, dashed through the centroid, the principal bending axes, the two directions in which a load bends the beam straight along itself. The bottom row shows the bent beams beside the straight ones. The I-section and the slotted tube are mirror-symmetric about the vertical, so their neutral axes lie square to the load and their tips move straight down. The turned I-section and the Z-section have no such symmetry: their neutral axes tilt, and their tips move down and sideways, towards the direction in which the section bends most easily.

In the matrix notation of beam theory the curvatures read as $\begin{pmatrix}\kappa_x\\ \kappa_y\end{pmatrix} = \frac{1}{E}\begin{pmatrix}I_{yy} & -I_{xy}\\ -I_{xy} & I_{xx}\end{pmatrix}^{-1}\begin{pmatrix}M_x\\ M_y\end{pmatrix}$, with $I_{xy}$ the product moment of area, which a mirror symmetry makes zero.

## 3. The Z-section with its load turning around

The load on the Z-section's tip turns once around the beam, in the plane of the section. Along either principal bending axis the tip moves along the load; between them it moves towards the axis about which the section bends most easily. Over the whole turn the tip goes around an ellipse whose axes are the principal axes.

In [ ]:
angles = np.linspace(0, 2 * np.pi, frames_count, endpoint=False)             # [frames]
loads = (mv.xy * (-angles / 2)).exp() >> (load * -up_direction)               # [frames] Point
triangles, boundary = sections[NAMES.index("Z-section")]                     # [triangles, 3], [boundary] Point
turning = cantilever(triangles, loads)
show_animation(render.animate_turning(triangles, boundary, loads, turning), "bending_turning", duration_ms)

## 4. Pushes along the beam, and the kern

A push along the beam through a point of the section is the line `beam_direction & point`, which is already a resultant, so the stiffness solves for its strain directly. A push through the centroid squeezes every fibre alike, and its neutral axis is the line at infinity. A push off the centroid also bends the section, and its neutral axis comes in from infinity on the far side of the centroid; where that line crosses the section, the fibres beyond it are stretched. The points through which a push leaves every fibre squeezed form the kern of the section. Solving the stiffness takes each push point to its neutral axis, a pairing of the points and lines of the section's plane called a polarity, and the pairing is symmetric: a pull through one point stretches the fibre through a second as much as a pull through the second stretches the fibre through the first.

In the notation of classical strength of materials the neutral axis of a push at $(e_x, e_y)$ in principal axes reads $1 + \frac{e_x x}{i_y^2} + \frac{e_y y}{i_x^2} = 0$, with $i_x, i_y$ the radii of gyration: the antipolar of the push point with respect to the central ellipse of inertia.

In [ ]:
def pulls(stiffness: Stiffness, points: Point) -> Strain:
    """The strain of a unit pull along the beam through each point of the section: its neutral axis,
    the point's polar line."""
    return stiffness.solve(beam_direction & points)                          # [...] Strain


def push_section(triangles: Point, points: Point, view: Point) -> tuple[Strain, Scalar, Scalar]:
    """Pushes along the beam through points of a section: their strains and the stresses at the corners of
    the section's triangles; and, for a unit pull through each point of the view, the stretch at those
    corners."""
    rigidity = stiffness(triangles, modulus)                                 # [] Resultant <- Strain
    strains = -push * pulls(rigidity, points)                                # [pushes] Strain
    stresses = stress(strains[..., None, None], modulus, triangles)          # [pushes, triangles, 3] Scalar
    pulled = pulls(rigidity, view)                                           # [rows, columns] Strain
    corners = stress(pulled[..., None, None], modulus, triangles)            # [rows, columns, triangles, 3] Scalar
    return strains, stresses, corners


points = point(push_points)                                                  # [pushes] Point
spots = np.linspace(-reach, reach, kern_grid)
view = point(np.stack(np.meshgrid(spots, spots), axis=-1))                   # [rows, columns] Point
pushes = [push_section(triangles, points, view) for triangles, _ in sections]
save_figure(render.draw_pushes(sections, points, view, pushes), "bending_kern");

The kern is drawn in purple, the push as a cross, and its neutral axis in green; the top row pushes through a point inside every kern, the bottom row through a point outside. Each corner of a section stays squeezed for push points on one side of a line, since its stress is linear in the push point, so the kern, where all corners stay squeezed, is a polygon: a diamond for the I-section, a tilted polygon for the Z-section. The figure finds it by checking every corner over a grid of push points. The kern of the slotted tube lies in its hollow, where a push can act through a plate on the beam's end; the slot moves the centroid to the left, and the kern is not round.